In [11]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.tmp.dir"] = "C:\\hadoop\\tmp"

In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("FixSocketIssue") \
    .master("local[1]") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .getOrCreate()


In [15]:
import os
from pyspark.sql import SparkSession

os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += ";C:\\hadoop\\bin"

spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.hadoop.io.native.lib.available","false") \
    .getOrCreate()

In [17]:
spark

In [19]:
df = spark.read.csv("credit_card_fraud_10k.csv", header=True, inferSchema=True)

In [21]:
df.show(truncate=False)

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|1             |84.47 |22              |Electronics      |0                  |0                |66                |3                |40            |0       |
|2             |541.82|3               |Travel           |1                  |0                |87                |1                |64            |0       |
|3             |237.01|17              |Grocery          |0                  |0                |49                |1                |61            |0       |
|4             |164.33|4               |Grocery     

In [23]:
from pyspark.sql.functions import col,count,when
from functools import reduce

null_condition = reduce(
    lambda x, y: x | y,
    [col(c).isNull() for c in df.columns]
)

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|             0|     0|               0|                0|                  0|                0|                 0|                0|             0|       0|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+



In [25]:
df_no_duplicates = df.dropDuplicates()
df.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|             1| 84.47|              22|      Electronics|                  0|                0|                66|                3|            40|       0|
|             2|541.82|               3|           Travel|                  1|                0|                87|                1|            64|       0|
|             3|237.01|              17|          Grocery|                  0|                0|                49|                1|            61|       0|
|             4|164.33|               4|          Gr

In [27]:
df= df.fillna({
    "merchant_category":"Unknown"
})

In [29]:
df_clean = df.withColumn(
    "high_amount_flag",
    when(col("amount") > 5000,1).otherwise(0)
)

df_clean.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|high_amount_flag|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+
|             1| 84.47|              22|      Electronics|                  0|                0|                66|                3|            40|       0|               0|
|             2|541.82|               3|           Travel|                  1|                0|                87|                1|            64|       0|               0|
|             3|237.01|              17|          Grocery|                  0|                0|                49|          

In [31]:
from pyspark.sql.functions import col, when

clean_df = (
    df
    .dropDuplicates()
    .dropna()
    .withColumn("amount_status",
                when((col("amount") >= 0) & (col("amount") <= 100000),
                     "VALID")
                .otherwise("INVALID"))
)

clean_df.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+-------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|amount_status|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+-------------+
|           498| 14.57|              22|      Electronics|                  0|                0|                55|                2|            35|       0|        VALID|
|           541|554.44|              22|           Travel|                  0|                0|                52|                1|            48|       0|        VALID|
|           667| 50.82|               3|         Clothing|                  0|                0|                63|                2|       

In [33]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("transaction_hour")

df_clean = df_clean.withColumn(
    "transactions_per_hour",
    count("transaction_id").over(windowSpec)
)
df_clean.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|high_amount_flag|transactions_per_hour|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+
|            48|132.14|               0|         Clothing|                  0|                0|                30|                2|            23|       0|               0|                  417|
|            52| 268.6|               0|             Food|                  0|                0|                85|                1|            34|       0|               0|                  417|
|            59

In [35]:
df_clean = df_clean.withColumn(
    "high_velocity_flag",
    when(col("velocity_last_24h") > 20,1).otherwise(0)
)


df_clean = df_clean.withColumn(
    "device_risk_flag",
    when(col("device_trust_score") < 0.3,1).otherwise(0)
)

df_clean = df_clean.withColumn(
    "location_risk_flag",
    when(col("location_mismatch") == 1,1).otherwise(0)
)

df_clean = df_clean.withColumn(
    "foreign_risk_flag",
    when(col("foreign_transaction") == 1,1).otherwise(0)
)


In [37]:
df_final = df_clean.withColumn(
    "fraud_risk_score",
    col("high_amount_flag") +
    col("high_velocity_flag") +
    col("device_risk_flag") +
    col("location_risk_flag") +
    col("foreign_risk_flag")
)

In [39]:
df_final.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|high_amount_flag|transactions_per_hour|high_velocity_flag|device_risk_flag|location_risk_flag|foreign_risk_flag|fraud_risk_score|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|            48|132.14|               0|         Clothing|                  0|                0|                30|                2|      

In [41]:
df_final.select(
    "transaction_id",
    "amount",
    "transaction_hour",
    "fraud_risk_score",
    "is_fraud"
).show()

+--------------+------+----------------+----------------+--------+
|transaction_id|amount|transaction_hour|fraud_risk_score|is_fraud|
+--------------+------+----------------+----------------+--------+
|             1| 84.47|              22|               0|       0|
|             2|541.82|               3|               1|       0|
|             3|237.01|              17|               0|       0|
|             4|164.33|               4|               1|       0|
|             5| 30.53|              15|               0|       0|
|             6| 30.53|              13|               0|       0|
|             7| 10.77|              18|               0|       0|
|             8|362.02|              13|               0|       0|
|             9|165.43|               8|               0|       0|
|            10|221.63|               5|               0|       0|
|            11|  3.74|              16|               0|       0|
|            12|630.64|              18|               0|     

In [43]:
pdf = df_final.toPandas()

pdf.to_csv(r"fraud_detection_pipeline_output.csv", index=False)

In [45]:
pdf.to_parquet(r"C:\Users\haris\Desktop\fraud_detection_pipeline_output.parquet", engine="pyarrow")

In [47]:
df = spark.read.csv(
    "fraud_detection_pipeline_output.csv",
    header=True,
    inferSchema=True
)

df.show()


+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|high_amount_flag|transactions_per_hour|high_velocity_flag|device_risk_flag|location_risk_flag|foreign_risk_flag|fraud_risk_score|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|            48|132.14|               0|         Clothing|                  0|                0|                30|                2|      

In [49]:
from pyspark.sql.functions import col

dim_time = df.select(
    col("transaction_hour").alias("time_id"),
    "transaction_hour"
).dropDuplicates()

dim_time.show()

+-------+----------------+
|time_id|transaction_hour|
+-------+----------------+
|     15|              15|
|      2|               2|
|     14|              14|
|     16|              16|
|     19|              19|
|      1|               1|
|     23|              23|
|     22|              22|
|      0|               0|
|      3|               3|
|      8|               8|
|     21|              21|
|     12|              12|
|     11|              11|
|     18|              18|
|     17|              17|
|      9|               9|
|      7|               7|
|      4|               4|
|     10|              10|
+-------+----------------+
only showing top 20 rows



In [51]:
pdf = dim_time.toPandas()

pdf.to_csv(r"dim_time.csv", index=False)

In [53]:
dim_location = df.select(
    col("location_mismatch").alias("location_id"),
    "location_mismatch",
    "foreign_transaction"
).dropDuplicates()

dim_location.show()

+-----------+-----------------+-------------------+
|location_id|location_mismatch|foreign_transaction|
+-----------+-----------------+-------------------+
|          0|                0|                  0|
|          1|                1|                  1|
|          1|                1|                  0|
|          0|                0|                  1|
+-----------+-----------------+-------------------+



In [55]:
pdf = dim_time.toPandas()

pdf.to_csv(r"dim_location.csv", index=False)

In [57]:
dim_cardholder = df.select(
    col("transaction_id").alias("cardholder_id"),
    "cardholder_age"
).dropDuplicates()

dim_cardholder.show()

+-------------+--------------+
|cardholder_id|cardholder_age|
+-------------+--------------+
|          396|            54|
|         3549|            53|
|         7938|            25|
|          809|            42|
|         4225|            39|
|         6724|            62|
|         2631|            33|
|         3986|            26|
|          424|            35|
|         1744|            63|
|         5954|            30|
|           65|            50|
|         1278|            40|
|         2898|            41|
|         8870|            65|
|         8951|            50|
|         1687|            21|
|         6046|            26|
|          590|            44|
|         3323|            51|
+-------------+--------------+
only showing top 20 rows



In [59]:
pdf = dim_time.toPandas()

pdf.to_csv(r"dim_cardholder.csv", index=False)

In [61]:
fact_transactions = df.select(
    "transaction_id",
    col("transaction_hour").alias("time_id"),
    col("location_mismatch").alias("location_id"),
    "amount",
    "fraud_risk_score",
    "is_fraud",
    "transactions_per_hour"
)

fact_transactions.show()

+--------------+-------+-----------+------+----------------+--------+---------------------+
|transaction_id|time_id|location_id|amount|fraud_risk_score|is_fraud|transactions_per_hour|
+--------------+-------+-----------+------+----------------+--------+---------------------+
|            48|      0|          0|132.14|               0|       0|                  417|
|            52|      0|          0| 268.6|               0|       0|                  417|
|            59|      0|          0|  8.33|               0|       0|                  417|
|           121|      0|          0|296.52|               0|       0|                  417|
|           146|      0|          0|  6.77|               0|       0|                  417|
|           170|      0|          0|160.88|               0|       0|                  417|
|           186|      0|          0|194.18|               0|       0|                  417|
|           258|      0|          0|214.35|               0|       0|           

In [63]:
pdf = dim_time.toPandas()

pdf.to_csv(r"fact_transactions.csv", index=False)

In [65]:
import pandas as pd

pdf = df.toPandas()

pdf.to_parquet("C:/Users/haris/test_parquet.parquet")

In [67]:
df = spark.read.parquet("C:/Users/haris/test_parquet.parquet")
df.show()

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|high_amount_flag|transactions_per_hour|high_velocity_flag|device_risk_flag|location_risk_flag|foreign_risk_flag|fraud_risk_score|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+----------------+---------------------+------------------+----------------+------------------+-----------------+----------------+
|            48|132.14|               0|         Clothing|                  0|                0|                30|                2|      

In [68]:
df = spark.read.csv(
    "bronze_layer/raw_transactions.csv",
    header=True,
    inferSchema=True
)

In [71]:
columns = [
    "transaction_id",
    "user_id",
    "amount",
    "transaction_time",
    "merchant",
    "location",
    "device_type",
    "transaction_status"
]

df = df.toDF(*columns)
df.show(5)

+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|        380637|   1383|    16|               0|       1|    0.78|          3|                40|
|        120902|    319|    18|               1|       1|    0.33|          4|                42|
|        636300|   1303|    11|               0|       1|    0.73|          3|                67|
|        202747|     69|    11|               0|       1|     0.6|          8|                33|
|        875016|   1722|    14|               1|       1|    0.62|          8|                35|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
only showing top 5 rows



In [77]:
df.show(truncate=True)

+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|        380637|   1383|    16|               0|       1|    0.78|          3|                40|
|        120902|    319|    18|               1|       1|    0.33|          4|                42|
|        636300|   1303|    11|               0|       1|    0.73|          3|                67|
|        202747|     69|    11|               0|       1|     0.6|          8|                33|
|        875016|   1722|    14|               1|       1|    0.62|          8|                35|
|        327820|   1106|    13|               0|       1|    0.73|          7|                20|
|        933023|    184|     5|               0|       1|    0.47|          7|                30|
|        818776|   1

In [79]:
from pyspark.sql.functions import col

# Remove duplicates
df_silver = df.dropDuplicates()

# Remove nulls (important columns)
df_silver = df_silver.filter(
    col("amount").isNotNull()
)

# Fix datatypes
df_silver = df_silver.withColumn(
    "amount", col("amount").cast("double")
)

df_silver.show()

+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|        429146|   1544|   0.0|               0|       0|    0.26|          6|                68|
|        310711|   1994|   0.0|               1|       1|    0.24|          5|                32|
|        224932|    838|   0.0|               1|       1|    0.55|          9|                53|
|        983917|    954|  20.0|               1|       0|    0.97|          3|                24|
|        649708|    471|  17.0|               1|       1|    0.69|          5|                24|
|        681340|   1676|   1.0|               1|       1|    0.31|          2|                51|
|        238144|    439|  15.0|               1|       1|    0.49|          3|                52|
|        895632|   1

In [81]:
import pandas as pd
df_bronze = df_silver.toPandas()


In [83]:
df_bronze.to_csv(
    "clean_transactions.csv",
    index=False
)

In [85]:
df_bronze = spark.createDataFrame(df_bronze)

In [87]:
from pyspark.sql.functions import col

df_silver = df_bronze.dropDuplicates()

df_silver = df_silver.filter(
    col("amount").isNotNull()
)

In [89]:
type(df_silver)

pyspark.sql.dataframe.DataFrame

In [91]:
df_silver.show(5)
df_silver.printSchema()

+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|        867434|     90|  19.0|               0|       0|    0.43|          9|                39|
|        721865|    367|  23.0|               0|       1|    0.75|          9|                20|
|        977655|    422|  14.0|               1|       1|    0.82|          6|                50|
|        585587|    666|   0.0|               0|       1|    0.41|          3|                29|
|        557598|   1336|   9.0|               0|       0|    0.86|          7|                57|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
only showing top 5 rows

root
 |-- transaction_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- am

In [92]:
df_pandas = df_silver.toPandas()

df_pandas.to_csv(
    "C:/Users/haris/fraud_project/data_lake/silver/clean_transactions.csv",
    index=False
)

In [93]:
df_gold = spark.createDataFrame(df_pandas)

In [97]:
from pyspark.sql.functions import when, col

df_gold = df_gold.withColumn(
    "fraud_flag",
    when(col("amount") > 5000, 1).otherwise(0)
)

df_gold.show()



+--------------+-------+------+----------------+--------+--------+-----------+------------------+----------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|fraud_flag|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+----------+
|        867434|     90|  19.0|               0|       0|    0.43|          9|                39|         0|
|        721865|    367|  23.0|               0|       1|    0.75|          9|                20|         0|
|        977655|    422|  14.0|               1|       1|    0.82|          6|                50|         0|
|        585587|    666|   0.0|               0|       1|    0.41|          3|                29|         0|
|        557598|   1336|   9.0|               0|       0|    0.86|          7|                57|         0|
|        461037|    403|  21.0|               1|       1|    0.12|          2|                41|         0|
|        465625|   

In [99]:
df_silver = spark.read.csv(
    "C:/Users/haris/fraud_project/data_lake/silver/clean_transactions.csv",
    header=True,
    inferSchema=True
)

df_silver.show(5)
df_silver.printSchema()

+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|transaction_id|user_id|amount|transaction_time|merchant|location|device_type|transaction_status|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
|        867434|     90|  19.0|               0|       0|    0.43|          9|                39|
|        721865|    367|  23.0|               0|       1|    0.75|          9|                20|
|        977655|    422|  14.0|               1|       1|    0.82|          6|                50|
|        585587|    666|   0.0|               0|       1|    0.41|          3|                29|
|        557598|   1336|   9.0|               0|       0|    0.86|          7|                57|
+--------------+-------+------+----------------+--------+--------+-----------+------------------+
only showing top 5 rows

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 

In [101]:
from pyspark.sql.functions import col, when

df_gold = df_silver.withColumn(
    "fraud_flag",
    when(col("amount") > 5000, 1).otherwise(0)
)

df_gold = df_gold.withColumn(
    "amount_category",
    when(col("amount") < 1000, "low")
    .when(col("amount") < 5000, "medium")
    .otherwise("high")
)

df_gold = df_gold.withColumn(
    "is_night_transaction",
    when(col("transaction_time") >= 20, 1).otherwise(0)
)

In [103]:
df_gold_pandas = df_gold.toPandas()

df_gold_pandas.to_csv(
    "C:/Users/haris/fraud_project/data_lake/gold/fraud_features.csv",
    index=False
)

In [105]:
df_gold.groupBy("fraud_flag").count().show()

+----------+-----+
|fraud_flag|count|
+----------+-----+
|         0| 4484|
+----------+-----+



In [277]:
df_gold.groupBy("amount_category").count().show()

+---------------+-----+
|amount_category|count|
+---------------+-----+
|            low| 2995|
+---------------+-----+

